# Credit Risk Prediction — Final Notebook

**Mục đích:** Notebook tổng hợp các thành phần hiện có của project theo một luồng chạy thống nhất trên Google Colab:

1. Chuẩn bị môi trường và tìm thư mục project.
2. Đọc dữ liệu và thực hiện EDA cơ bản.
3. Làm sạch dữ liệu, chia train/test.
4. Tạo và lưu các cấu hình feature preprocessing (`.npy`, `.json`, `.joblib`).
5. Huấn luyện và đánh giá Logistic Regression, SVM, Random Forest.
6. Kiểm tra các deliverables được tạo ra.

> Notebook sử dụng các hàm trong `modules/data_processing.py` và `modules/model_training.py`, không viết lại toàn bộ logic trong notebook.

## 0. Environment setup

Notebook hỗ trợ hai trường hợp:

- Mở notebook trực tiếp từ GitHub bằng Google Colab: thư mục repository thường đã được tải vào runtime.
- Nếu chưa có thư mục project, notebook sẽ clone repository từ `REPO_URL`.

Nếu repository của nhóm đã đổi tên hoặc URL khác, sửa `REPO_URL` bên dưới cho đúng.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/DuyTuong101/credit-risk-prediction.git"
REPO_DIR_NAME = "credit-risk-prediction"

def is_project_root(path: Path) -> bool:
    return (
        (path / "modules" / "data_processing.py").is_file()
        and (path / "modules" / "model_training.py").is_file()
        and (path / "data").is_dir()
    )

# Tìm project trong working directory hiện tại và các thư mục thường gặp trên Colab.
candidate_dirs = [
    Path.cwd(),
    Path.cwd() / REPO_DIR_NAME,
    Path("/content") / REPO_DIR_NAME,
    Path("/content") / "credit-risk-prediction-main",
]

PROJECT_ROOT = next((p.resolve() for p in candidate_dirs if is_project_root(p)), None)

# Nếu notebook chưa nằm trong một bản checkout của project thì clone repository.
if PROJECT_ROOT is None:
    if not REPO_URL or "<" in REPO_URL:
        raise RuntimeError("Hãy cập nhật REPO_URL thành URL GitHub thật của repository.")
    clone_path = Path("/content") / REPO_DIR_NAME
    if clone_path.exists():
        subprocess.run(["git", "-C", str(clone_path), "pull"], check=True)
    else:
        subprocess.run(["git", "clone", REPO_URL, str(clone_path)], check=True)
    PROJECT_ROOT = clone_path.resolve()

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Project files:", sorted(os.listdir(PROJECT_ROOT))[:20])

# Cài các thư viện được khai báo trong requirements.txt.
requirements_file = PROJECT_ROOT / "requirements.txt"
if requirements_file.is_file():
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_file)],
        check=True
    )
else:
    raise FileNotFoundError("Không tìm thấy requirements.txt trong project root.")

## 1. Import libraries and configure paths

Các đường dẫn được tạo từ `PROJECT_ROOT`, vì vậy không phụ thuộc notebook đang chạy ở thư mục nào.

In [ ]:
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from modules.data_processing import (
    audit_constraints,
    clean_raw_data,
    split_data,
    build_preprocessing_pipeline,
    get_feature_names,
    save_features,
)
from modules.model_training import train_and_evaluate

warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid")

DATA_RAW_DIR = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
FEATURES_DIR = PROJECT_ROOT / "features"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"

for folder in [DATA_RAW_DIR, DATA_PROCESSED_DIR, FEATURES_DIR, MODELS_DIR, REPORTS_DIR, FIGURES_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RAW_DATA_PATH = DATA_RAW_DIR / "credit_risk_dataset.csv"
CLEAN_DATA_PATH = DATA_PROCESSED_DIR / "credit_risk_cleaned.csv"

if not RAW_DATA_PATH.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy dataset tại {RAW_DATA_PATH}. "
        "Hãy bảo đảm file data/raw/credit_risk_dataset.csv đã được commit lên GitHub."
    )

print("Raw dataset:", RAW_DATA_PATH)
print("Features directory:", FEATURES_DIR)
print("Reports directory:", REPORTS_DIR)

## 2. Load raw dataset

Dataset được lấy trực tiếp từ `data/raw/credit_risk_dataset.csv` trong repository. Không cần mount Google Drive.

In [ ]:
df_raw = pd.read_csv(RAW_DATA_PATH)

print("Dataset shape:", df_raw.shape)
print("\nColumn names:")
print(df_raw.columns.tolist())

display(df_raw.head())

## 3. Exploratory Data Analysis (EDA)

Phần này tạo các thống kê chính để kiểm tra dữ liệu trước khi preprocessing: kiểu dữ liệu, missing values, duplicates, phân phối target và một số biểu đồ.

In [ ]:
print("=== DATA TYPES ===")
display(df_raw.dtypes.to_frame("data_type"))

print("=== MISSING VALUES ===")
missing_summary = pd.DataFrame({
    "missing_count": df_raw.isna().sum(),
    "missing_percent": (df_raw.isna().mean() * 100).round(2),
})
display(missing_summary[missing_summary["missing_count"] > 0])

print("=== DUPLICATES ===")
print("Duplicated rows:", int(df_raw.duplicated().sum()))

print("=== NUMERICAL SUMMARY ===")
display(df_raw.describe().T)

In [ ]:
plt.figure(figsize=(6, 4))
ax = sns.countplot(data=df_raw, x="loan_status")
ax.set_title("Target distribution: loan_status")
ax.set_xlabel("loan_status (0 = non-default, 1 = default)")
ax.set_ylabel("Number of samples")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "final_target_distribution.png", dpi=200)
plt.show()

target_counts = df_raw["loan_status"].value_counts(normalize=True).sort_index() * 100
display(target_counts.rename("percentage").to_frame())

In [ ]:
numeric_cols_for_eda = df_raw.select_dtypes(include="number").columns.tolist()
plt.figure(figsize=(10, 7))
sns.heatmap(df_raw[numeric_cols_for_eda].corr(), cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "final_correlation_heatmap.png", dpi=200)
plt.show()

In [ ]:
print("=== BUSINESS / LOGICAL CONSTRAINT AUDIT ===")
display(audit_constraints(df_raw).to_frame("rows_violating"))

## 4. Data cleaning and train/test split

Sử dụng `clean_raw_data()` từ module. Sau đó chia train/test theo stratified sampling để giữ tỷ lệ hai lớp gần tương đương giữa hai tập.

In [ ]:
df_clean = clean_raw_data(df_raw, verbose=True)
df_clean.to_csv(CLEAN_DATA_PATH, index=False)

X_train, X_test, y_train, y_test = split_data(
    df_clean,
    target_col="loan_status",
    test_size=0.2,
    random_state=42,
)

print("Cleaned dataset:", df_clean.shape)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Train default rate:", round(float(np.mean(y_train)), 4))
print("Test default rate:", round(float(np.mean(y_test)), 4))

## 5. Feature preprocessing and extraction

Notebook chạy các cấu hình đã được định nghĩa trong notebook preprocessing hiện tại. Mỗi pipeline chỉ được `fit` trên training set; test set chỉ được `transform` để tránh data leakage.

Các feature arrays và metadata được lưu vào thư mục `features/`.

In [ ]:
num_cols = [
    "person_age",
    "person_income",
    "person_emp_length",
    "loan_amnt",
    "loan_int_rate",
    "loan_percent_income",
    "cb_person_cred_hist_length",
]
ordinal_cols = ["loan_grade"]
nominal_cols = [
    "person_home_ownership",
    "loan_intent",
    "cb_person_default_on_file",
]

configs = {
    "baseline": dict(impute_strategy="median", scaler_type="standard"),
    "minmax_0_1": dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1)),
    "minmax_m1_1": dict(impute_strategy="median", scaler_type="minmax", feature_range=(-1, 1)),
    "robust_logincome": dict(impute_strategy="median", scaler_type="robust", log_cols=["person_income"]),
    "knn_indicator": dict(impute_strategy="knn", scaler_type="standard", add_missing_indicator=True),
    "standard_pca90": dict(impute_strategy="median", scaler_type="standard", pca_variance=0.90),
    "standard_pca95": dict(impute_strategy="median", scaler_type="standard", pca_variance=0.95),
    "minmax_pca95": dict(impute_strategy="median", scaler_type="minmax", feature_range=(0, 1), pca_variance=0.95),
}

feature_summary = []

for config_name, config in configs.items():
    print(f"Processing configuration: {config_name}")

    preprocessing_pipeline = build_preprocessing_pipeline(
        num_cols=num_cols,
        ordinal_cols=ordinal_cols,
        nominal_cols=nominal_cols,
        **config,
    )

    X_train_transformed = preprocessing_pipeline.fit_transform(X_train)
    X_test_transformed = preprocessing_pipeline.transform(X_test)

    assert np.isfinite(X_train_transformed).all(), f"Non-finite values in {config_name} train"
    assert np.isfinite(X_test_transformed).all(), f"Non-finite values in {config_name} test"

    save_features(
        name=config_name,
        pipeline=preprocessing_pipeline,
        X_train_t=X_train_transformed,
        X_test_t=X_test_transformed,
        y_train=y_train,
        y_test=y_test,
        config=config,
        out_dir=str(FEATURES_DIR),
        save_pipeline=(config_name == "baseline"),
    )

    feature_summary.append({
        "configuration": config_name,
        "imputer": config.get("impute_strategy"),
        "scaler": config.get("scaler_type"),
        "pca_variance": config.get("pca_variance"),
        "train_features": X_train_transformed.shape[1],
        "test_features": X_test_transformed.shape[1],
    })

feature_summary_df = pd.DataFrame(feature_summary)
display(feature_summary_df)

## 6. Train and evaluate traditional machine learning models

Sử dụng `train_and_evaluate()` trong `modules/model_training.py` để so sánh ba mô hình:

- Logistic Regression
- Support Vector Machine (SVM)
- Random Forest

Kết quả cross-validation, test metrics, confusion matrix, classification report và model tốt nhất được lưu vào `reports/` và `models/`.

**Lưu ý về cấu trúc code hiện tại:** `modules/model_training.py` đang tự xây dựng preprocessing pipeline bên trong hàm `train_and_evaluate()`. Vì vậy, bước huấn luyện dưới đây hiện chạy trên DataFrame đã làm sạch và tự preprocess trong module; nó chưa trực tiếp đọc các file `X_train_*.npy` được tạo ở bước 5. Bước 5 và bước 6 đều được thực thi trong notebook, nhưng việc nối trực tiếp feature arrays vào model training là phần nên đồng bộ tiếp ở lần hoàn thiện sau để tránh hai luồng preprocessing song song.

In [ ]:
training_results = train_and_evaluate(
    df=df_clean,
    target_col="loan_status",
    test_size=0.2,
    cv_folds=3,
    random_state=42,
    scaler="standard",
    numeric_imputation="median",
    categorical_imputation="most_frequent",
    model_names=("logistic_regression", "svm", "random_forest"),
    models_dir=MODELS_DIR,
    reports_dir=REPORTS_DIR,
)

print("=== CROSS-VALIDATION MODEL COMPARISON ===")
display(training_results["comparison"])

print("=== HELD-OUT TEST METRICS ===")
display(pd.Series(training_results["test_metrics"], name="value").to_frame())

print("Best model saved at:", training_results["model_path"])

## 7. Validate generated deliverables

Kiểm tra nhanh các file `.npy`, JSON, model và báo cáo. Nếu cell này chạy thành công, các file đầu ra chính đã được tạo.

In [ ]:
# Validate X/y arrays for all preprocessing configurations.
y_train_saved = np.load(FEATURES_DIR / "y_train.npy")
y_test_saved = np.load(FEATURES_DIR / "y_test.npy")

validation_rows = []
for config_name in configs:
    xtr_path = FEATURES_DIR / f"X_train_{config_name}.npy"
    xte_path = FEATURES_DIR / f"X_test_{config_name}.npy"

    Xtr_saved = np.load(xtr_path)
    Xte_saved = np.load(xte_path)

    assert Xtr_saved.shape[0] == len(y_train_saved)
    assert Xte_saved.shape[0] == len(y_test_saved)
    assert Xtr_saved.shape[1] == Xte_saved.shape[1]

    validation_rows.append({
        "configuration": config_name,
        "X_train_shape": Xtr_saved.shape,
        "X_test_shape": Xte_saved.shape,
        "valid": True,
    })

display(pd.DataFrame(validation_rows))

required_outputs = [
    MODELS_DIR / "best_model.joblib",
    REPORTS_DIR / "model_comparison.csv",
    REPORTS_DIR / "test_metrics.json",
    REPORTS_DIR / "classification_report.json",
    REPORTS_DIR / "confusion_matrix.csv",
]

for output_path in required_outputs:
    print(("OK  " if output_path.is_file() else "MISSING  ") + str(output_path.relative_to(PROJECT_ROOT)))

assert all(path.is_file() for path in required_outputs), "Some expected model/report outputs are missing."
print("\nDeliverable validation completed.")

## 8. Final output summary

Các file được sinh ra nằm trong repository:

- `data/processed/credit_risk_cleaned.csv`
- `features/`: train/test feature arrays, labels, configuration metadata, feature names và baseline preprocessor.
- `models/best_model.joblib`
- `reports/`: model comparison, test metrics, classification report, confusion matrix và EDA figures.

**Quan trọng:** Google Colab runtime là môi trường tạm thời. Sau khi chạy xong, cần tải các file kết quả cần nộp về máy hoặc commit/push chúng lên repository theo quy định của nhóm. Notebook không mount Google Drive.